# Convolutional Neural Networks (CNN)
## Plant Disease Detection using PlantVillage Dataset
This notebook uses the `tensorflow_datasets` library to load the PlantVillage dataset, which contains images of healthy and diseased leaves.

In [ ]:
# If tensorflow_datasets is not installed, uncomment and run the following line:
# !pip install tensorflow_datasets

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Rescaling

In [ ]:
# Load the PlantVillage dataset from TensorFlow Datasets
# We split it into 80% training and 20% validation
dataset_name = 'plant_village'
(ds_train, ds_val), ds_info = tfds.load(
    dataset_name,
    split=['train[:80%]', 'train[80%:]'],
    with_info=True,
    as_supervised=True  # Returns tuple (image, label) instead of dictionary
)

num_classes = ds_info.features['label'].num_classes
class_names = ds_info.features['label'].names

print(f"Number of classes: {num_classes}")
print(f"Sample classes: {class_names[:5]}")

In [ ]:
# Preprocess the data: Resize and Batch
IMG_SIZE = 128
BATCH_SIZE = 32

def format_image(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, label

# Apply formatting, batching, and prefetching for performance optimization
train_batches = ds_train.map(format_image).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
val_batches = ds_val.map(format_image).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
# Visualize some training examples
plt.figure(figsize=(10, 10))
for images, labels in train_batches.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        # Images are cast back to int for proper displaying with imshow
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Build the CNN Model
model = Sequential([
    # Rescaling layer to normalize pixel values to [0, 1]
    Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.summary()

In [ ]:
# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# Train the model
# Note: Training might take some time depending on hardware.
# Adjust epochs lower if needed for quicker testing.
epochs = 10
history = model.fit(
    train_batches,
    validation_data=val_batches,
    epochs=epochs
)

In [ ]:
# Plot Training & Validation Accuracy and Loss
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss over Epochs')
plt.legend()

plt.show()